In [1]:
import torch
from torchinfo import summary

from models import VanillaVAE

import os
import yaml
import argparse
import numpy as np
from pathlib import Path

import torch
import torch.backends.cudnn as cudnn

from models import VanillaVAE
from experiment import VAEXperiment

from pytorch_lightning import Trainer, seed_everything
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.strategies import DDPStrategy
from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint

from dataset import VAEDataset

In [2]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        try:
            _ = torch.zeros(1, device="mps")
            return torch.device("mps")
        except Exception:
            pass
    return torch.device("cpu")

device = get_device()
print("Using device:", device)

Using device: cuda


In [3]:
def test_vanilla_vae(device):
    # Instantiate model and move to device
    model = VanillaVAE(3, 10).to(device)
    # Tell summary which device to use
    print(summary(model, input_size=(1, 3, 64, 64), device=device))
    # Forward pass test
    print("\n=== Forward Test ===")
    x = torch.randn(16, 3, 64, 64, device=device)
    out = model(x)
    print("Output[0] shape:", out[0].shape)

    # Loss test
    print("\n=== Loss Test ===")
    loss = model.loss_function(*out, M_N=0.005)
    print("Loss:", loss)

In [4]:
test_vanilla_vae(device=device)

Layer (type:depth-idx)                   Output Shape              Param #
VanillaVAE                               [1, 3, 64, 64]            --
├─Sequential: 1-1                        [1, 512, 2, 2]            --
│    └─Sequential: 2-1                   [1, 32, 32, 32]           --
│    │    └─Conv2d: 3-1                  [1, 32, 32, 32]           896
│    │    └─BatchNorm2d: 3-2             [1, 32, 32, 32]           64
│    │    └─LeakyReLU: 3-3               [1, 32, 32, 32]           --
│    └─Sequential: 2-2                   [1, 64, 16, 16]           --
│    │    └─Conv2d: 3-4                  [1, 64, 16, 16]           18,496
│    │    └─BatchNorm2d: 3-5             [1, 64, 16, 16]           128
│    │    └─LeakyReLU: 3-6               [1, 64, 16, 16]           --
│    └─Sequential: 2-3                   [1, 128, 8, 8]            --
│    │    └─Conv2d: 3-7                  [1, 128, 8, 8]            73,856
│    │    └─BatchNorm2d: 3-8             [1, 128, 8, 8]            256
│   

In [5]:
config_path = "configs/vae.yaml"   # set directly
with open(config_path, "r") as f:
    config = yaml.safe_load(f)
tb_logger = TensorBoardLogger(
    save_dir=config["logging_params"]["save_dir"],
    name=config["model_params"]["name"],
)

In [7]:

# For reproducibility
seed_everything(config['exp_params']['manual_seed'], True)

# ----------------------
# Device / accelerator logic
# ----------------------
trainer_params = config['trainer_params'].copy()

# "gpus" can be an int (e.g., 1) or a list (e.g., [0, 1])
raw_gpus = trainer_params.pop("gpus", 0)

if isinstance(raw_gpus, (list, tuple)):
    num_gpus = len(raw_gpus)
    devices = list(raw_gpus)        # pass the list directly to Trainer
else:
    num_gpus = int(raw_gpus) if raw_gpus else 0
    devices = num_gpus if num_gpus > 0 else 1   # will be overridden for CPU branch anyway

if torch.cuda.is_available() and num_gpus > 0:
    accelerator = "gpu"
    strategy = DDPStrategy(find_unused_parameters=False) if num_gpus > 1 else None
    cudnn.benchmark = True
    print(f"Using CUDA with {num_gpus} GPU(s): {devices}")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    # Apple Silicon (M1/M2/M3) path
    accelerator = "mps"
    devices = 1
    strategy = None
    print("Using Apple MPS device.")
else:
    accelerator = "cpu"
    devices = 1
    strategy = None
    print("Using CPU.")

# ----------------------
# Model & experiment
# ----------------------
model = VanillaVAE[config['model_params']['name']](**config['model_params'])
experiment = VAEXperiment(model, config['exp_params'])

# pin_memory only makes sense when using CUDA
use_pin_memory = torch.cuda.is_available()
data = VAEDataset(**config["data_params"], pin_memory=use_pin_memory)
data.setup()

# ----------------------
# Trainer
# ----------------------
trainer_kwargs = dict(
    logger=tb_logger,
    callbacks=[
        LearningRateMonitor(),
        ModelCheckpoint(
            save_top_k=2,
            dirpath=os.path.join(tb_logger.log_dir, "checkpoints"),
            monitor="val_loss",
            save_last=True,
        ),
    ],
    accelerator=accelerator,
    devices=devices,
    **trainer_params,
)

if strategy is not None:
    trainer_kwargs["strategy"] = strategy

runner = Trainer(**trainer_kwargs)

# ----------------------
# Folders & run
# ----------------------
Path(f"{tb_logger.log_dir}/Samples").mkdir(exist_ok=True, parents=True)
Path(f"{tb_logger.log_dir}/Reconstructions").mkdir(exist_ok=True, parents=True)

print(f"======= Training {config['model_params']['name']} =======")
runner.fit(experiment, datamodule=data)

Seed set to 1265


Using CUDA with 1 GPU(s): [3]


TypeError: type 'VanillaVAE' is not subscriptable